In [8]:
!pip install requests beautifulsoup4 pdfplumber sentence-transformers faiss-cpu


In [9]:
import requests
from bs4 import BeautifulSoup
import json
import time
import re
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

In [10]:
places = {
    "Luxor": "Luxor",
    "Aswan": "Aswan",
    "Alexandria": "Alexandria",
    "Hurghada": "Hurghada",
    "Sharm El Sheikh": "Sharm_el-Sheikh",
    "Dahab": "Dahab",
    "Siwa Oasis": "Siwa_Oasis",
    "Marsa Alam": "Marsa_Alam",
    "Valley of the Kings": "Valley_of_the_Kings",
    "Karnak Temple": "Karnak"
}

In [11]:
def clean_text(text):
    text = re.sub(r"\[\d+\]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def get_text(title):
    url = f"https://en.wikipedia.org/wiki/{title}"

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        print(f"Error: {title}")
        return None

    soup = BeautifulSoup(response.text, "html.parser")
    paragraphs = soup.find_all("p")

    text = " ".join([p.get_text() for p in paragraphs])
    return clean_text(text)

In [20]:
dataset = []

for place, wiki_title in places.items():
    print(f"Scraping: {place} ...")

    text = get_text(wiki_title)

    if text:
        dataset.append({
            "place": place,
            "city": place,
            "type": "tourism",
            "text": text,
            "source": f"https://en.wikipedia.org/wiki/{wiki_title}"
        })

        print(f"{place} DONE")

    time.sleep(1)


with open("egypt_places.json", "w", encoding="utf-8") as f:
    json.dump(dataset, f, ensure_ascii=False, indent=4)

print("Dataset saved ")

Scraping: Luxor ...
Luxor DONE
Scraping: Aswan ...
Aswan DONE
Scraping: Alexandria ...
Alexandria DONE
Scraping: Hurghada ...
Hurghada DONE
Scraping: Sharm El Sheikh ...
Sharm El Sheikh DONE
Scraping: Dahab ...
Dahab DONE
Scraping: Siwa Oasis ...
Siwa Oasis DONE
Scraping: Marsa Alam ...
Marsa Alam DONE
Scraping: Valley of the Kings ...
Valley of the Kings DONE
Scraping: Karnak Temple ...
Karnak Temple DONE
Dataset saved ✔️


In [13]:
def chunk_text(text, chunk_size=100, overlap=20):
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)

    return chunks

In [14]:
chunked_data = []

for item in dataset:
    chunks = chunk_text(item["text"])

    for i, chunk in enumerate(chunks):
        chunked_data.append({
            "place": item["place"],
            "city": item["city"],
            "type": item["type"],
            "chunk_id": i,
            "text": chunk,
            "source": item["source"]
        })


with open("chunked_egypt_places.json", "w", encoding="utf-8") as f:
    json.dump(chunked_data, f, ensure_ascii=False, indent=4)

print("Chunking done")

Chunking done ✔️


In [15]:


model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [item["text"] for item in chunked_data]

embeddings = model.encode(texts, show_progress_bar=True)

for i, item in enumerate(chunked_data):
    item["embedding"] = embeddings[i].tolist()


with open("embedded_egypt_places.json", "w", encoding="utf-8") as f:
    json.dump(chunked_data, f, ensure_ascii=False, indent=4)

print("Embeddings saved")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/14 [00:00<?, ?it/s]

Embeddings saved ✔️


In [18]:
embeddings_array = np.array(embeddings).astype("float32")

dimension = embeddings_array.shape[1]
index = faiss.IndexFlatL2(dimension)

index.add(embeddings_array)

print("FAISS index created")

FAISS index created ✔️


In [19]:
import pickle

faiss.write_index(index, "faiss_index.index")

with open("faiss_metadata.pkl", "wb") as f:
    pickle.dump(chunked_data, f)

print("FAISS + Metadata saved")

FAISS + Metadata saved ✔️
